In [1]:
# Sumativa 3 – Fase 3  
Optimización del OEE mediante Mantenimiento Predictivo  

## Integrantes:
- Alana Bermúdez  
- Gonzalo Mansilla  
- Eduardo Villalón  

## Curso: MCDI500  

SyntaxError: invalid syntax (2563818269.py, line 2)

## Contexto y base del proyecto (Fase 3)

## Propósito del notebook

Este notebook se construye a partir del trabajo desarrollado en la Fase 3 (Formativa 3), donde se implementaron algoritmos recursivos y análisis de complejidad utilizando el dataset procesado en Fase 2.

En esta etapa (Sumativa 3), se extiende dicho trabajo incorporando una estructura más robusta basada en programación estructurada y Programación Orientada a Objetos (POO), junto con validaciones y análisis de eficiencia.

Este enfoque permite avanzar desde un prototipo funcional hacia una solución más estructurada, reutilizable y escalable para el análisis de datos.

## Estructura del flujo y funciones principales

En esta sección se reorganiza el flujo del notebook mediante funciones estructuradas que permiten separar responsabilidades como la carga de datos, validación y preparación de información.

Este enfoque facilita la reutilización del código y su posterior extensión mediante Programación Orientada a Objetos.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

In [ ]:
def obtener_raiz_proyecto() -> Path:
    """
    Identifica la raíz del proyecto según la ubicación desde donde se ejecuta el notebook.
    Permite ejecutar el notebook desde la carpeta F3 o desde la raíz del repositorio.
    """
    cwd = Path.cwd()

    if (cwd / "data").exists():
        return cwd

    if (cwd.parent / "data").exists():
        return cwd.parent

    raise FileNotFoundError("No se pudo identificar la raíz del proyecto.")


PROJECT_ROOT = obtener_raiz_proyecto()
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed" / "dataset_procesado.csv"

print("Raíz del proyecto:", PROJECT_ROOT)
print("Dataset procesado:", DATA_PROCESSED)

In [ ]:
def cargar_dataset_procesado(ruta: Path) -> pd.DataFrame:
    """
    Carga el dataset procesado de Fase 2 y convierte la columna date a formato datetime.
    
    Parámetros:
        ruta (Path): ruta relativa o absoluta del archivo CSV procesado.
    
    Retorna:
        pd.DataFrame: dataset cargado y preparado para validación inicial de Fase 3.
    """
    if not ruta.exists():
        raise FileNotFoundError(f"No se encontró el archivo: {ruta}")

    df = pd.read_csv(ruta, parse_dates=["date"])

    print(f"Dataset cargado correctamente: {df.shape[0]} filas y {df.shape[1]} columnas.")
    return df


df = cargar_dataset_procesado(DATA_PROCESSED)
df.head()

In [ ]:
def validar_dataset_f3(df: pd.DataFrame) -> None:
    """
    Valida condiciones mínimas del dataset procesado para continuar con la Fase 3.
    No reemplaza el pipeline de Fase 2; solo confirma que el insumo está listo para análisis algorítmico.
    """
    columnas_esperadas = (
        ["date", "device", "failure"] +
        [f"metric{i}" for i in range(1, 10)] +
        ["year", "month", "day"]
    )

    columnas_faltantes = [col for col in columnas_esperadas if col not in df.columns]

    assert len(columnas_faltantes) == 0, f"Faltan columnas esperadas: {columnas_faltantes}"
    assert df.shape[0] == 124493, f"Cantidad de filas inesperada: {df.shape[0]}"
    assert df.isnull().sum().sum() == 0, "Existen valores nulos en el dataset."
    assert df.duplicated().sum() == 0, "Existen registros duplicados en el dataset."
    assert pd.api.types.is_datetime64_any_dtype(df["date"]), "La columna date no está en formato datetime."
    assert set(df["failure"].unique()).issubset({0, 1}), "La variable failure contiene valores distintos de 0 y 1."

    print("Validaciones automáticas F3: OK")


validar_dataset_f3(df)

In [ ]:
def resumen_dataset_f3(df: pd.DataFrame) -> pd.DataFrame:
    """
    Genera un resumen técnico del dataset utilizado como insumo para Fase 3.
    """
    resumen = pd.DataFrame({
        "criterio": [
            "Filas",
            "Columnas",
            "Valores nulos",
            "Duplicados",
            "Fecha mínima",
            "Fecha máxima",
            "Dispositivos únicos",
            "Eventos de falla",
            "Registros sin falla"
        ],
        "resultado": [
            df.shape[0],
            df.shape[1],
            int(df.isnull().sum().sum()),
            int(df.duplicated().sum()),
            df["date"].min(),
            df["date"].max(),
            df["device"].nunique(),
            int(df["failure"].sum()),
            int((df["failure"] == 0).sum())
        ]
    })

    return resumen


resumen_f3 = resumen_dataset_f3(df)
resumen_f3

## Validación inicial del dataset para Fase 3

El dataset utilizado en esta fase corresponde al archivo procesado en Fase 2. No se reconstruye el pipeline de limpieza, ya que el objetivo de la Formativa 3 es avanzar hacia el diseño de algoritmos, recursividad y mediciones de complejidad.

La validación inicial confirma que el dataset mantiene las condiciones esperadas: ausencia de valores nulos, ausencia de duplicados, columna `date` en formato temporal, presencia de variables derivadas `year`, `month` y `day`, y variable objetivo binaria `failure`.

Estas condiciones permiten continuar con las actividades de los otros integrantes del equipo: implementación de algoritmos estructurados, comparación de eficiencia y desarrollo de un algoritmo recursivo.

In [ ]:
SRC_PATH = PROJECT_ROOT / "F3" / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

#from carga_validacion import cargar_dataset_procesado, validar_dataset_f3

#df_prueba = cargar_dataset_procesado(DATA_PROCESSED)
#validar_dataset_f3(df_prueba)

print("Importación desde script F3: OK")

In [ ]:
import pandas as pd
from pathlib import Path

# Ruta directa al dataset procesado
ruta = Path.cwd().parent / "data" / "processed" / "dataset_procesado.csv"

df_prueba = pd.read_csv(ruta)

print("Dataset cargado correctamente ✅")
df_prueba.head()


## Algoritmo Recursivo y Análisis de Complejidad

In [ ]:
# Seleccionar columna metric1 y crear muestra de 5000 datos
datos_metric1 = df_prueba["metric1"].head(5000).tolist()

print("Cantidad de elementos en la muestra:", len(datos_metric1))

In [ ]:
# Función merge: combina dos listas ordenadas
def merge(izquierda, derecha):
    resultado = []
    i = j = 0
    
    # Comparar elementos de ambas listas
    while i < len(izquierda) and j < len(derecha):
        if izquierda[i] < derecha[j]:
            resultado.append(izquierda[i])
            i += 1
        else:
            resultado.append(derecha[j])
            j += 1
    
    # Agregar elementos restantes
    resultado.extend(izquierda[i:])
    resultado.extend(derecha[j:])
    
    return resultado

In [ ]:
# Función recursiva Merge Sort
def merge_sort(lista):
    
    # Caso base: lista de 1 o 0 elementos ya está ordenada
    if len(lista) <= 1:
        return lista
    
    # Dividir en dos mitades
    medio = len(lista) // 2
    izquierda = lista[:medio]
    derecha = lista[medio:]
    
    # Llamada recursiva a cada mitad
    izquierda_ordenada = merge_sort(izquierda)
    derecha_ordenada = merge_sort(derecha)
    
    # Combinar usando merge
    return merge(izquierda_ordenada, derecha_ordenada)

In [ ]:
# Ejecutar Merge Sort sobre la muestra
ordenado_merge = merge_sort(datos_metric1)

# Mostrar los primeros 10 valores ordenados
print("Primeros 10 valores (Merge Sort):")
print(ordenado_merge[:10])

In [ ]:
# Validar con función nativa de Python
ordenado_python = sorted(datos_metric1)

# Comparar ambos resultados
assert ordenado_merge == ordenado_python

print("✅ Validación exitosa: ambos resultados son iguales")

In [ ]:
import timeit

# Tiempo de ejecución Merge Sort
tiempo_merge = timeit.timeit(
    lambda: merge_sort(datos_metric1.copy()),
    number=5
)

# Tiempo de ejecución sorted()
tiempo_python = timeit.timeit(
    lambda: sorted(datos_metric1.copy()),
    number=5
)

print("Tiempo Merge Sort:", tiempo_merge)
print("Tiempo sorted():", tiempo_python)

In [ ]:
import pandas as pd

resultados = pd.DataFrame({
    "Algoritmo": ["Merge Sort (Recursivo)", "sorted() (Optimizado)"],
    "Tiempo de ejecución": [tiempo_merge, tiempo_python]
})

resultados

### Interpretación de Resultados

La recursividad es una técnica de programación en la cual una función se llama a sí misma para resolver un problema dividiéndolo en subproblemas más pequeños. En este caso, el algoritmo Merge Sort utiliza este enfoque para ordenar los datos de manera eficiente.

Merge Sort funciona bajo el paradigma de "dividir y conquistar", separando la lista en mitades sucesivas hasta llegar a listas de un solo elemento, las cuales se consideran ordenadas por definición. Luego, estas listas se combinan mediante la función merge, que construye una lista ordenada final.

Desde el punto de vista de complejidad, Merge Sort tiene una complejidad temporal de O(n log n), lo que lo hace eficiente incluso para grandes volúmenes de datos. Sin embargo, su implementación recursiva implica un mayor uso de memoria debido a la pila de llamadas.

Al comparar con la función nativa sorted(), se observa que esta última presenta un mejor rendimiento, ya que está optimizada internamente en Python.

### Conclusión

La implementación de Merge Sort permitió aplicar de manera práctica el concepto de recursividad, evidenciando cómo un problema complejo puede resolverse mediante su descomposición en partes más pequeñas. Este enfoque resulta especialmente útil desde una perspectiva formativa, ya que facilita la comprensión de algoritmos de ordenamiento eficientes.

No obstante, en un contexto aplicado al análisis de datos, se observa que el uso de funciones optimizadas como sorted() ofrece mejores tiempos de ejecución y menor sobrecarga de recursos. Por ello, si bien la recursividad es conceptualmente poderosa, su uso en entornos productivos debe evaluarse cuidadosamente, priorizando soluciones que combinen eficiencia, legibilidad y escalabilidad.

## Implementación de flujo estructurado

En esta sección se reorganiza el proceso del análisis mediante un enfoque estructurado basado en funciones, permitiendo separar claramente las etapas de carga, validación y preparación de datos.

Este diseño facilita la mantenibilidad del código y su posterior integración con Programación Orientada a Objetos.

Este enfoque permite además una mejor trazabilidad del flujo de datos, facilitando su análisis y validación en etapas posteriores del proyecto.

In [ ]:
def cargar_datos(ruta):
    import pandas as pd
    
    try:
        df = pd.read_csv(ruta)
        print("✅ Dataset cargado correctamente:", df.shape)
        return df
    except Exception as e:
        print("❌ Error al cargar dataset:", e)

In [ ]:
def validar_datos(df):
    assert df is not None, "El dataset está vacío"
    assert df.isnull().sum().sum() == 0, "Existen valores nulos"
    assert df.duplicated().sum() == 0, "Existen duplicados"
    
    print("✅ Validación completada")

In [ ]:
def preparar_datos(df):
    datos = df["metric1"].head(5000).tolist()
    print("✅ Datos preparados:", len(datos))
    return datos

In [ ]:
def main():
    df = cargar_datos(DATA_PROCESSED)
    validar_datos(df)
    datos = preparar_datos(df)
    
    return df, datos


### Ejecución del flujo principal

La función main() permite coordinar las distintas etapas del flujo estructurado, integrando la carga, validación y preparación del dataset.

Su ejecución permite generar tanto el dataset limpio como la muestra de datos utilizada en el análisis algorítmico.

In [ ]:
df_main, datos_main = main()

In [ ]:
df_main.head()

### Visualización del dataset

Se muestran los primeros registros del dataset procesado, verificando que la carga de datos y el flujo estructurado se ejecutaron correctamente.

Esto permite confirmar la integridad de los datos y la correcta ejecución del flujo implementado.

In [ ]:
class Preprocesador:
    def __init__(self, df):
        self._df = df
    
    def obtener_datos(self):
        return self._df

In [ ]:
prep = Preprocesador(df_main)
prep.obtener_datos().head()

### Implementación inicial de POO

Se implementa una clase básica que encapsula el dataset, permitiendo organizar el acceso a los datos mediante métodos.

Este enfoque establece una base para la extensión del proyecto mediante principios de Programación Orientada a Objetos.

Esta implementación permite una futura extensión del sistema mediante herencia y reutilización de componentes dentro del proceso de análisis.

In [2]:
import pandas as pd
import numpy as np
print("Librerías cargadas correctamente.")

Librerías cargadas correctamente.


In [3]:
class Preprocesador:
    def __init__(self, df):
        # El guion bajo (_df) es una convención para indicar que es un atributo "protegido" (Encapsulamiento)
        self._df = df

In [4]:
class Transformador(Preprocesador):
    """
    Clase derivada que extiende a Preprocesador mediante herencia.
    """
    def __init__(self, df):
        super().__init__(df)
        
    def imputar_nulos(self, columna, metodo='mean'):
        """Imputa nulos asignando el resultado directamente."""
        if metodo == 'mean':
            valor = self._df[columna].mean()
        else:
            valor = self._df[columna].median()
        # Cambiamos fillna(inplace=True) por una asignación directa
        self._df[columna] = self._df[columna].fillna(valor)
        return self

    def escalar_minmax(self, columnas):
        """Escala valores asignando el resultado directamente."""
        for col in columnas:
            min_val = self._df[col].min()
            max_val = self._df[col].max()
            self._df[col] = (self._df[col] - min_val) / (max_val - min_val)
        return self

In [5]:
# Crear un DataFrame pequeño con datos nulos
data = pd.DataFrame({'a': [1, 5, np.nan], 'b': [10, 50, 100]})

# Usar tu clase Transformador
t = Transformador(data)
t.imputar_nulos('a', 'mean').escalar_minmax(['a', 'b'])

# Imprimir el resultado
print(t._df)

     a         b
0  0.0  0.000000
1  1.0  0.444444
2  0.5  1.000000
